# RAG Evaluation with LangSmith

This notebook demonstrates how to build a complete RAG (Retrieval-Augmented Generation) evaluation pipeline using LangSmith.

**Stack:**
- LLM: Groq (Llama 3.3 70B)
- Embeddings: HuggingFace `BAAI/bge-small-en-v1.5` (free, local)
- Evaluation Framework: LangSmith
- Data Source: Web-based documents from Lilian Weng's blog

**Evaluators:**
1. Correctness: Compares response against reference answer
2. Relevance: Checks if response addresses the question
3. Groundedness: Ensures response doesn't hallucinate outside retrieved documents
4. Retrieval Relevance: Validates that retrieved documents are relevant to query

## Step 1: Build the RAG Pipeline


In [60]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

# List of URLs to load documents from
urls = [
    "https://lilianweng.github.io/posts/2023-06-23-agent/",
    "https://lilianweng.github.io/posts/2023-03-15-prompt-engineering/",
    "https://lilianweng.github.io/posts/2023-10-25-adv-attack-llm/",
]

# Load documents from the URLs
docs = [WebBaseLoader(url).load() for url in urls]
docs_list = [item for sublist in docs for item in sublist]

# Initialize a text splitter with specified chunk size and overlap
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=250, chunk_overlap=0
)

# Split the documents into chunks
doc_splits = text_splitter.split_documents(docs_list)

# Free local embeddings
embedding = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

# Add the document chunks to the vector store
vectorstore = InMemoryVectorStore.from_documents(
    documents=doc_splits,
    embedding=embedding,
)

# Create a retriever component
retriever = vectorstore.as_retriever(k=6)

## Step 2: Create the RAG Bot

Build a RAG bot that retrieves relevant documents and generates answers using an LLM.

In [61]:
from langchain.chat_models import init_chat_model
from langsmith import traceable
llm = init_chat_model("groq:llama-3.3-70b-versatile", api_key=os.getenv("GROQ_API_KEY"))


@traceable()
def rag_bot(question: str) -> dict:
    # Retrieve relevant context
    docs = retriever.invoke(question)
    docs_string = " ".join(doc.page_content for doc in docs)

    instructions = f"""You are a helpful assistant who is good at \
analyzing source information and answering questions. \
Use the following source documents to answer the user's questions. \
If you don't know the answer, just say that you don't know. \
Use three sentences maximum and keep the answer concise.

Documents:
{docs_string}"""

    # Generate answer
    ai_msg = llm.invoke([
        {"role": "system", "content": instructions},
        {"role": "user", "content": question},
    ])
    return {"answer": ai_msg.content, "documents": docs}

## Step 3: Create the RAG Evaluation Dataset

Define test cases with expected reference answers to evaluate RAG performance.

In [62]:
from langsmith import Client

client = Client()

examples = [
    {
        "inputs": {
            "question": "How does the ReAct agent use self-reflection?"
        },
        "outputs": {
            "answer": "ReAct integrates reasoning and acting, performing "
            "actions - such tools like Wikipedia search API - and then "
            "observing / reasoning about the tool outputs."
        },
    },
    {
        "inputs": {
            "question": "What are the types of biases that can arise "
            "with few-shot prompting?"
        },
        "outputs": {
            "answer": "The biases that can arise with few-shot prompting "
            "include (1) Majority label bias, (2) Recency bias, and "
            "(3) Common token bias."
        },
    },
    {
        "inputs": {
            "question": "What are five types of adversarial attacks?"
        },
        "outputs": {
            "answer": "Five types of adversarial attacks are "
            "(1) Token manipulation, (2) Gradient based attack, "
            "(3) Jailbreak prompting, (4) Human red-teaming, "
            "(5) Model red-teaming."
        },
    },
]

dataset_name = "RAG Test Evaluation"

if not client.has_dataset(dataset_name=dataset_name):
    dataset = client.create_dataset(dataset_name=dataset_name)
    client.create_examples(dataset_id=dataset.id, examples=examples)
else:
    dataset = client.read_dataset(dataset_name=dataset_name)

## Step 4: Define RAG-Specific Evaluators

Implement four evaluators to assess RAG quality across different dimensions.

### Evaluator 1: Correctness (Response vs Reference Answer)

Grade student answers based on factual accuracy relative to ground truth, ensuring no conflicting statements.

In [63]:
from typing_extensions import Annotated, TypedDict
from langchain_groq import ChatGroq

# Structured output schema
class CorrectnessGrade(TypedDict):
    explanation: Annotated[
        str, ..., "Explain your reasoning for the score"
    ]
    correct: Annotated[
        bool, ..., "True if the answer is correct, False otherwise."
    ]

correctness_instructions = """You are a teacher grading a quiz.

You will be given a QUESTION, the GROUND TRUTH (correct) ANSWER, \
and the STUDENT ANSWER.

Here is the grade criteria to follow:
(1) Grade the student answers based ONLY on their factual accuracy \
relative to the ground truth answer.
(2) Ensure that the student answer does not contain any conflicting \
statements.
(3) It is OK if the student answer contains more information than \
the ground truth answer, as long as it is factually accurate \
relative to the ground truth answer.

Correctness:
A correctness value of True means that the student's answer meets \
all of the criteria.
A correctness value of False means that the student's answer does \
not meet all of the criteria.

Explain your reasoning in a step-by-step manner to ensure your \
reasoning and conclusion are correct.

Avoid simply stating the correct answer at the outset.
Respond in JSON format."""

grader_llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key=os.getenv("GROQ_API_KEY"),
    temperature=0
).with_structured_output(
    CorrectnessGrade, method="json_mode", strict=True
)

def correctness(
    inputs: dict, outputs: dict, reference_outputs: dict
) -> bool:
    """An evaluator for RAG answer accuracy"""
    answers = f"""\
QUESTION: {inputs['question']}
GROUND TRUTH ANSWER: {reference_outputs['answer']}
STUDENT ANSWER: {outputs['answer']}"""

    grade = grader_llm.invoke([
        {"role": "system", "content": correctness_instructions},
        {"role": "user", "content": answers},
    ])
    return grade["correctness"]

### Evaluator 2: Relevance (Response vs Input)

Check if the answer is concise, relevant, and directly addresses the question.

In [64]:
class RelevanceGrade(TypedDict):
    explanation: Annotated[
        str, ..., "Explain your reasoning for the score"
    ]
    relevant: Annotated[
        bool, ...,
        "Provide the score on whether the answer addresses the question",
    ]

relevance_instructions = """You are a teacher grading a quiz.

You will be given a QUESTION and a STUDENT ANSWER.

Here is the grade criteria to follow:
(1) Ensure the STUDENT ANSWER is concise and relevant to the QUESTION
(2) Ensure the STUDENT ANSWER helps to answer the QUESTION

Relevance:
A relevance value of True means that the student's answer meets \
all of the criteria.
A relevance value of False means that the student's answer does \
not meet all of the criteria.

Explain your reasoning in a step-by-step manner to ensure your \
reasoning and conclusion are correct.

Avoid simply stating the correct answer at the outset.
Respond in JSON format."""

relevance_llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key=os.getenv("GROQ_API_KEY"),
    temperature=0
).with_structured_output(
    RelevanceGrade, method="json_mode", strict=True
)

def relevance(inputs: dict, outputs: dict) -> bool:
    """A simple evaluator for RAG answer helpfulness."""
    answer = (
        f"QUESTION: {inputs['question']}\n"
        f"STUDENT ANSWER: {outputs['answer']}"
    )
    grade = relevance_llm.invoke([
        {"role": "system", "content": relevance_instructions},
        {"role": "user", "content": answer},
    ])
    return grade["relevance"]

### Evaluator 3: Groundedness (Response vs Retrieved Documents)

Ensure the answer is grounded in retrieved facts and doesn't hallucinate information outside document scope.

In [65]:
class GroundedGrade(TypedDict):
    explanation: Annotated[
        str, ..., "Explain your reasoning for the score"
    ]
    grounded: Annotated[
        bool, ...,
        "Provide the score on if the answer hallucinates from the documents",
    ]

grounded_instructions = """You are a teacher grading a quiz.

You will be given FACTS and a STUDENT ANSWER.

Here is the grade criteria to follow:
(1) Ensure the STUDENT ANSWER is grounded in the FACTS.
(2) Ensure the STUDENT ANSWER does not contain "hallucinated" \
information outside the scope of the FACTS.

Grounded:
A grounded value of True means that the student's answer meets \
all of the criteria.
A grounded value of False means that the student's answer does \
not meet all of the criteria.

Explain your reasoning in a step-by-step manner to ensure your \
reasoning and conclusion are correct.

Avoid simply stating the correct answer at the outset.
Respond in JSON format."""

grounded_llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key=os.getenv("GROQ_API_KEY"),
    temperature=0
).with_structured_output(
    GroundedGrade, method="json_mode", strict=True
)

def groundedness(inputs: dict, outputs: dict) -> bool:
    """A simple evaluator for RAG answer groundedness."""
    doc_string = "\n\n".join(
        doc.page_content for doc in outputs["documents"]
    )
    answer = f"FACTS: {doc_string}\nSTUDENT ANSWER: {outputs['answer']}"
    grade = grounded_llm.invoke([
        {"role": "system", "content": grounded_instructions},
        {"role": "user", "content": answer},
    ])
    return grade["grounded"]

### Evaluator 4: Retrieval Relevance (Retrieved Docs vs Input)

Validate that retrieved documents contain keywords or semantic meaning related to the question.

In [66]:
class RetrievalRelevanceGrade(TypedDict):
    explanation: Annotated[
        str, ..., "Explain your reasoning for the score"
    ]
    relevant: Annotated[
        bool, ...,
        "True if the retrieved documents are relevant to the question, "
        "False otherwise",
    ]

retrieval_relevance_instructions = """You are a teacher grading a quiz.

You will be given a QUESTION and a set of FACTS provided by the student.

Here is the grade criteria to follow:
(1) Your goal is to identify FACTS that are completely unrelated \
to the QUESTION
(2) If the facts contain ANY keywords or semantic meaning related \
to the question, consider them relevant
(3) It is OK if the facts have SOME information that is unrelated \
to the question as long as (2) is met

Relevance:
A relevance value of True means that the FACTS contain ANY keywords \
or semantic meaning related to the QUESTION and are therefore relevant.
A relevance value of False means that the FACTS are completely \
unrelated to the QUESTION.

Explain your reasoning in a step-by-step manner to ensure your \
reasoning and conclusion are correct.

Avoid simply stating the correct answer at the outset.
Respond in JSON format."""

retrieval_relevance_llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key=os.getenv("GROQ_API_KEY"),
    temperature=0
).with_structured_output(
    RetrievalRelevanceGrade, method="json_mode", strict=True
)

def retrieval_relevance(inputs: dict, outputs: dict) -> bool:
    """An evaluator for document relevance"""
    doc_string = "\n\n".join(
        doc.page_content for doc in outputs["documents"]
    )
    answer = f"FACTS: {doc_string}\nQUESTION: {inputs['question']}"
    grade = retrieval_relevance_llm.invoke([
        {"role": "system", "content": retrieval_relevance_instructions},
        {"role": "user", "content": answer},
    ])
    return grade["relevance"]

## Step 5: Run the Complete RAG Evaluation

Execute all evaluators on the test dataset and analyze results.

In [67]:
def target(inputs: dict) -> dict:
    return rag_bot(inputs["question"])

experiment_results = client.evaluate(
    target,
    data=dataset_name,
    evaluators=[
        correctness,
        groundedness,
        relevance,
        retrieval_relevance,
    ],
    experiment_prefix="rag-doc-relevance",
    metadata={"version": "LCEL context, groq-llama3-groq-70b-8192-tool-use-preview"},
)

# Explore results locally as a dataframe
experiment_results.to_pandas()

View the evaluation results for experiment: 'rag-doc-relevance-c3f84c20' at:
https://smith.langchain.com/o/6d419d79-1f69-49f1-9409-d187df487fdf/datasets/bf33c75e-9589-4dc1-b9f2-d0e839e4b4d9/compare?selectedSessions=094791b8-9a9a-441c-ab1e-d3501edf4c3f




0it [00:00, ?it/s]

,inputs.question,outputs.answer,outputs.documents,error,reference.answer,feedback.correctness,feedback.groundedness,feedback.relevance,feedback.retrieval_relevance,execution_time,example_id,id
0,What are five types of adversarial attacks?,"According to the source, five approaches to fi...",[page_content='Black-box attacks assume that a...,None,Five types of adversarial attacks are (1) Toke...,True,True,True,True,0.555887,43670579-fef6-4a9b-81e7-c12b52af2f97,019ea235-c39f-72e3-9573-9c863a891be6
1,How does the ReAct agent use self-reflection?,The ReAct agent uses self-reflection to refine...,[page_content='Self-reflection is a vital aspe...,None,"ReAct integrates reasoning and acting, perform...",True,True,True,True,0.438347,bedcbe23-a82e-4518-8952-82dc518f9d45,019ea235-d0ff-7812-af30-6f2891bed10c
2,What are the types of biases that can arise wi...,I don't know the specific types of biases that...,[page_content='Zero-shot and few-shot learning...,None,The biases that can arise with few-shot prompt...,False,True,False,False,0.339087,e9f3c2c9-b6bf-4e7b-a182-d5da99453b84,019ea235-dcb1-7d13-8b53-afd34a319fae
